# HW06-ICA Part B :: Binary Classification Metrics

COSC 6373 -- Adam Nelson-Archer, 2140122

This notebook follows the Part B tasks using transfer learning with ResNet50 on Horses vs Camels.


## Setup and imports

In [ ]:
from __future__ import annotations

import os
import random
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    auc,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_curve,
)
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

SEED = 6373
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU available:", len(tf.config.list_physical_devices("GPU")) > 0)
print("Working directory:", os.getcwd())


## Dataset paths and split helpers

Expected extracted dataset layout:

- `DATA_ROOT/train/<horse-class-folder>/...`
- `DATA_ROOT/train/<camel-class-folder>/...`
- `DATA_ROOT/test/<horse-class-folder>/...`
- `DATA_ROOT/test/<camel-class-folder>/...`

If your class folder names are not exactly `horses` / `camels`, this code still maps labels using name matching.


In [ ]:
NOTEBOOK_DIR = Path.cwd()
DATA_ROOT = (NOTEBOOK_DIR / "data" / "horses_camels").resolve()

TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"

print("DATA_ROOT:", DATA_ROOT)
print("TRAIN_DIR exists:", TRAIN_DIR.exists())
print("TEST_DIR exists:", TEST_DIR.exists())

if not TRAIN_DIR.exists() or not TEST_DIR.exists():
    raise FileNotFoundError(
        "Dataset not found. Put extracted Horses vs Camels data under "
        "HW6/Part_B/data/horses_camels with train/ and test/ folders."
    )


In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def infer_label_from_path(path: Path) -> int:
    """Map horse->0, camel->1 using folder name text."""
    token = path.parent.name.lower()
    if "horse" in token:
        return 0
    if "camel" in token:
        return 1
    raise ValueError(f"Cannot infer class label from folder name: {path.parent.name}")


def collect_image_paths(root_dir: Path) -> pd.DataFrame:
    rows = []
    for p in root_dir.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            y = infer_label_from_path(p)
            rows.append({"path": str(p), "label": int(y)})
    if not rows:
        raise ValueError(f"No images found under: {root_dir}")
    df = pd.DataFrame(rows)
    return df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)


train_all_df = collect_image_paths(TRAIN_DIR)
test_df = collect_image_paths(TEST_DIR)

print("Total train images:", len(train_all_df))
print("Total test images:", len(test_df))
print("Train class counts:\n", train_all_df["label"].value_counts().sort_index())
print("Test class counts:\n", test_df["label"].value_counts().sort_index())


In [ ]:
# Assignment requirement: build validation set from original training set,
# with 20 images per class.
val_n_per_class = 20

if (train_all_df["label"].value_counts() < val_n_per_class).any():
    raise ValueError("Not enough training images to sample 20 per class for validation.")

val_df = (
    train_all_df.groupby("label", group_keys=False)
    .apply(lambda g: g.sample(n=val_n_per_class, random_state=SEED))
    .reset_index(drop=True)
)

# Remove validation images from original training set by path.
train_df = train_all_df[~train_all_df["path"].isin(set(val_df["path"]))].reset_index(drop=True)


def balance_binary_df(df: pd.DataFrame, seed: int = SEED) -> pd.DataFrame:
    """Undersample to the minority class count so each split is balanced."""
    counts = df["label"].value_counts()
    if len(counts) != 2:
        raise ValueError(f"Expected 2 classes, found: {counts.to_dict()}")
    n = int(counts.min())
    out = (
        df.groupby("label", group_keys=False)
        .apply(lambda g: g.sample(n=n, random_state=seed))
        .sample(frac=1.0, random_state=seed)
        .reset_index(drop=True)
    )
    return out


# Validation is already exactly 20 per class, but we keep this explicit.
val_df = balance_binary_df(val_df)
train_df = balance_binary_df(train_df)
test_df = balance_binary_df(test_df)

print("Train split size:", len(train_df))
print("Validation split size:", len(val_df))
print("Test split size:", len(test_df))
print("\nTrain class counts:\n", train_df["label"].value_counts().sort_index())
print("\nValidation class counts:\n", val_df["label"].value_counts().sort_index())
print("\nTest class counts:\n", test_df["label"].value_counts().sort_index())


## Input pipeline

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 8
LR = 1e-4


def _load_and_preprocess(path: tf.Tensor, label: tf.Tensor):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    img = preprocess_input(img)
    return img, label


def make_dataset(df: pd.DataFrame, one_hot: bool, training: bool) -> tf.data.Dataset:
    paths = df["path"].astype(str).values
    labels = df["label"].astype(np.int32).values
    if one_hot:
        labels = tf.one_hot(labels, depth=2)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(buffer_size=len(df), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(_load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds


## Model builders

In [ ]:
def build_bce_model(freeze_base: bool) -> tf.keras.Model:
    base = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3), pooling="avg")
    base.trainable = not freeze_base

    inputs = tf.keras.Input(shape=(224, 224, 3))
    # Keep BN/internals frozen only when base is frozen; allow full updates otherwise.
    x = base(inputs, training=not freeze_base)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = tf.keras.Model(inputs, outputs, name=f"resnet50_bce_{'frozen' if freeze_base else 'full'}")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


def build_cce_model(freeze_base: bool) -> tf.keras.Model:
    base = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3), pooling="avg")
    base.trainable = not freeze_base

    inputs = tf.keras.Input(shape=(224, 224, 3))
    # Keep BN/internals frozen only when base is frozen; allow full updates otherwise.
    x = base(inputs, training=not freeze_base)
    outputs = layers.Dense(2, activation="softmax")(x)
    model = tf.keras.Model(inputs, outputs, name=f"resnet50_cce_{'frozen' if freeze_base else 'full'}")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


## Evaluation utilities (confusion matrix, precision, accuracy, sensitivity, specificity, ROC/AUC)

In [ ]:
@dataclass
class EvalResult:
    accuracy: float
    precision: float
    sensitivity: float
    specificity: float
    auc_score: float
    cm: np.ndarray
    fpr: np.ndarray
    tpr: np.ndarray


def collect_predictions(model: tf.keras.Model, ds: tf.data.Dataset, mode: str):
    y_true_batches = []
    y_prob_batches = []

    for x_batch, y_batch in ds:
        y_hat = model.predict(x_batch, verbose=0)
        if mode == "bce":
            y_true = tf.cast(tf.reshape(y_batch, [-1]), tf.int32).numpy()
            y_prob = tf.reshape(y_hat, [-1]).numpy()
        elif mode == "cce":
            y_true = tf.argmax(y_batch, axis=1).numpy()
            y_prob = y_hat[:, 1]
        else:
            raise ValueError(f"Unknown mode: {mode}")

        y_true_batches.append(y_true)
        y_prob_batches.append(y_prob)

    y_true = np.concatenate(y_true_batches)
    y_prob = np.concatenate(y_prob_batches)
    y_pred = (y_prob >= 0.5).astype(np.int32)
    return y_true, y_prob, y_pred


def evaluate_binary(y_true: np.ndarray, y_prob: np.ndarray, y_pred: np.ndarray) -> EvalResult:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    sensitivity = recall_score(y_true, y_pred, zero_division=0)
    specificity = tn / max((tn + fp), 1)

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_score = auc(fpr, tpr)

    return EvalResult(
        accuracy=float(accuracy),
        precision=float(precision),
        sensitivity=float(sensitivity),
        specificity=float(specificity),
        auc_score=float(auc_score),
        cm=cm,
        fpr=fpr,
        tpr=tpr,
    )


def plot_learning_curve(history: tf.keras.callbacks.History, title: str):
    plt.figure(figsize=(6, 4))
    plt.plot(history.history["loss"], marker="o", label="train loss")
    plt.plot(history.history["val_loss"], marker="o", label="val loss")
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()


def plot_cm(cm: np.ndarray, title: str):
    fig, ax = plt.subplots(figsize=(4, 4))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["horse", "camel"])
    disp.plot(cmap="Blues", ax=ax, colorbar=False)
    ax.set_title(title)
    plt.tight_layout()


def plot_roc(fpr: np.ndarray, tpr: np.ndarray, auc_score: float, title: str):
    plt.figure(figsize=(5, 4))
    plt.plot(fpr, tpr, label=f"AUC={auc_score:.4f}")
    plt.plot([0, 1], [0, 1], "k--", alpha=0.7)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.legend(loc="lower right")
    plt.tight_layout()


## Run Model 1: BCE + sigmoid (frozen base)

- Dense(1, sigmoid)
- Binary cross-entropy
- Freeze all layers except classifier


In [ ]:
train_ds_bce = make_dataset(train_df, one_hot=False, training=True)
val_ds_bce = make_dataset(val_df, one_hot=False, training=False)
test_ds_bce = make_dataset(test_df, one_hot=False, training=False)

model1_bce_frozen = build_bce_model(freeze_base=True)
history1 = model1_bce_frozen.fit(
    train_ds_bce,
    validation_data=val_ds_bce,
    epochs=EPOCHS,
    verbose=1,
)

plot_learning_curve(history1, "Model 1 (BCE frozen): loss vs epochs")
y_true_1, y_prob_1, y_pred_1 = collect_predictions(model1_bce_frozen, test_ds_bce, mode="bce")
eval1 = evaluate_binary(y_true_1, y_prob_1, y_pred_1)

plot_cm(eval1.cm, "Model 1 (BCE frozen) confusion matrix")
plot_roc(eval1.fpr, eval1.tpr, eval1.auc_score, "Model 1 (BCE frozen) ROC")

print(eval1)


## Run Model 2: CCE + softmax (frozen base)

- Reinitialize with ImageNet weights
- Dense(2, softmax)
- One-hot labels + categorical cross-entropy
- Freeze all layers except classifier


In [ ]:
train_ds_cce = make_dataset(train_df, one_hot=True, training=True)
val_ds_cce = make_dataset(val_df, one_hot=True, training=False)
test_ds_cce = make_dataset(test_df, one_hot=True, training=False)

model2_cce_frozen = build_cce_model(freeze_base=True)
history2 = model2_cce_frozen.fit(
    train_ds_cce,
    validation_data=val_ds_cce,
    epochs=EPOCHS,
    verbose=1,
)

plot_learning_curve(history2, "Model 2 (CCE frozen): loss vs epochs")
y_true_2, y_prob_2, y_pred_2 = collect_predictions(model2_cce_frozen, test_ds_cce, mode="cce")
eval2 = evaluate_binary(y_true_2, y_prob_2, y_pred_2)

plot_cm(eval2.cm, "Model 2 (CCE frozen) confusion matrix")
plot_roc(eval2.fpr, eval2.tpr, eval2.auc_score, "Model 2 (CCE frozen) ROC")

print(eval2)


## Run Model 3A: full fine-tuning with BCE + sigmoid

- Reinitialize with ImageNet weights
- Train full model (no frozen layers)


In [ ]:
model3a_bce_full = build_bce_model(freeze_base=False)
history3a = model3a_bce_full.fit(
    train_ds_bce,
    validation_data=val_ds_bce,
    epochs=EPOCHS,
    verbose=1,
)

plot_learning_curve(history3a, "Model 3A (BCE full): loss vs epochs")
y_true_3a, y_prob_3a, y_pred_3a = collect_predictions(model3a_bce_full, test_ds_bce, mode="bce")
eval3a = evaluate_binary(y_true_3a, y_prob_3a, y_pred_3a)

plot_cm(eval3a.cm, "Model 3A (BCE full) confusion matrix")
plot_roc(eval3a.fpr, eval3a.tpr, eval3a.auc_score, "Model 3A (BCE full) ROC")

print(eval3a)


## Run Model 3B: full fine-tuning with CCE + softmax

- Reinitialize with ImageNet weights
- Train full model (no frozen layers)


In [ ]:
model3b_cce_full = build_cce_model(freeze_base=False)
history3b = model3b_cce_full.fit(
    train_ds_cce,
    validation_data=val_ds_cce,
    epochs=EPOCHS,
    verbose=1,
)

plot_learning_curve(history3b, "Model 3B (CCE full): loss vs epochs")
y_true_3b, y_prob_3b, y_pred_3b = collect_predictions(model3b_cce_full, test_ds_cce, mode="cce")
eval3b = evaluate_binary(y_true_3b, y_prob_3b, y_pred_3b)

plot_cm(eval3b.cm, "Model 3B (CCE full) confusion matrix")
plot_roc(eval3b.fpr, eval3b.tpr, eval3b.auc_score, "Model 3B (CCE full) ROC")

print(eval3b)


## Metrics summary table

In [ ]:
summary_df = pd.DataFrame(
    [
        {"model": "M1 BCE frozen", **eval1.__dict__},
        {"model": "M2 CCE frozen", **eval2.__dict__},
        {"model": "M3A BCE full", **eval3a.__dict__},
        {"model": "M3B CCE full", **eval3b.__dict__},
    ]
)

summary_view = summary_df[["model", "accuracy", "precision", "sensitivity", "specificity", "auc_score"]]
summary_view = summary_view.sort_values(by="accuracy", ascending=False).reset_index(drop=True)
summary_view


## Short comparison paragraph (fill after running)

BCE vs CCE (frozen classifier only):
- TODO: compare convergence speed, stability of validation loss, and final metrics.

BCE vs CCE (full fine-tuning):
- TODO: compare convergence speed, stability of validation loss, and final metrics.

Overall takeaway:
- TODO: summarize which setup worked best and why.


## Written questions

### What is a confusion matrix?
A confusion matrix is a table that compares true labels to predicted labels. For binary classification, it gives true negatives (TN), false positives (FP), false negatives (FN), and true positives (TP).

### What is Accuracy and how is it measured?
Accuracy is the proportion of correct predictions among all predictions: `(TP + TN) / (TP + TN + FP + FN)`.

### What is Precision and how is it measured?
Precision is the proportion of predicted positives that are actually positive: `TP / (TP + FP)`.

### What is Sensitivity and how is it measured?
Sensitivity (Recall, True Positive Rate) measures how many actual positives are correctly detected: `TP / (TP + FN)`.

### What is Specificity and how is it measured?
Specificity (True Negative Rate) measures how many actual negatives are correctly detected: `TN / (TN + FP)`.

### What is an ROC curve and how is it computed?
The ROC curve plots True Positive Rate vs False Positive Rate at many decision thresholds. It is computed by sweeping thresholds over predicted probabilities and calculating TPR/FPR at each threshold. AUC summarizes the curve into one score.

### When is it best to use a softmax versus a sigmoid activation function in the last layer?
Use sigmoid when each output is an independent binary probability (binary or multi-label classification). Use softmax when classes are mutually exclusive and outputs should form a probability distribution summing to 1.


## Acknowledgment

I used a coding assistant (ChatGPT in Cursor, GPT-5.2) to help scaffold and organize this notebook.
